In [1]:
from pathlib import Path
import json

import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'odev3').is_dir():
    if PROJECT_ROOT.name == 'odev3':
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise FileNotFoundError('Notebook proje kökünden veya odev3 klasöründen çalıştırılmalıdır.')
ODEV3 = PROJECT_ROOT / 'odev3'
CORPORA = ('cremad', 'meld')
print(f'Proje kökü: {PROJECT_ROOT}')
print('Veri kümeleri: CREMA-D, MELD')
print('Protokol: model seçimi validation; test yalnız nihai değerlendirme için kullanılır.')

Proje kökü: C:\Users\ahmet\Desktop\470 proje
Veri kümeleri: CREMA-D, MELD
Protokol: model seçimi validation; test yalnız nihai değerlendirme için kullanılır.


In [2]:
# Her veri kümesinde seçilmiş tek validation konfigürasyonunu göster.
validation_rows = []
for corpus in CORPORA:
    frame = pd.read_csv(ODEV3 / 'outputs' / corpus / 'validation_results.csv')
    selected = frame[frame['selected'].astype(str).str.lower().eq('true')]
    assert len(selected) == 1, f'{corpus}: tam bir seçilmiş koşu bekleniyordu'
    row = selected.iloc[0]
    validation_rows.append({
        'dataset': 'CREMA-D' if corpus == 'cremad' else 'MELD',
        'trial': int(row['trial']),
        'stage': row['search_stage'],
        'seed': int(row['seed']),
        'batch_size': int(row['batch_size']),
        'learning_rate': row['learning_rate'],
        'patience': int(row['patience']),
        'hidden_dims': row['hidden_dims'],
        'activation': row['activation'],
        'batch_norm': bool(row['batch_norm']),
        'dropout': row['dropout'],
        'weight_decay': row['weight_decay'],
        'best_epoch': int(row['best_epoch']),
        'val_accuracy': row['val_accuracy'],
        'val_balanced_accuracy': row['val_balanced_accuracy'],
        'val_macro_f1': row['val_macro_f1'],
        'val_weighted_f1': row['val_weighted_f1'],
    })
best_validation = pd.DataFrame(validation_rows)
best_validation

,dataset,trial,stage,seed,batch_size,learning_rate,patience,hidden_dims,activation,batch_norm,dropout,weight_decay,best_epoch,val_accuracy,val_balanced_accuracy,val_macro_f1,val_weighted_f1
0,CREMA-D,20,refinement,42,128,0.0003,8,512-256-128,relu,True,0.3,0.0001,29,0.485366,0.48373,0.484510,0.485993
1,MELD,15,screening,42,64,0.0003,8,512-256,relu,True,0.5,0.0001,3,0.261084,0.27279,0.252545,0.250498


In [3]:
# Kaydedilmiş örnek tahminlerinden test metriklerini bağımsız yeniden hesapla.
metric_functions = {
    'accuracy': lambda y, p: accuracy_score(y, p),
    'balanced_accuracy': lambda y, p: balanced_accuracy_score(y, p),
    'macro_f1': lambda y, p: f1_score(y, p, average='macro'),
    'weighted_f1': lambda y, p: f1_score(y, p, average='weighted'),
}
test_rows = []
for corpus in CORPORA:
    corpus_dir = ODEV3 / 'outputs' / corpus
    predictions = pd.read_csv(corpus_dir / 'test_predictions.csv')
    result = json.loads((corpus_dir / 'result.json').read_text(encoding='utf-8'))
    y_true = predictions['label_idx'].to_numpy()
    y_pred = predictions['predicted_idx'].to_numpy()
    for metric, function in metric_functions.items():
        recomputed = float(function(y_true, y_pred))
        reported = float(result['test'][metric])
        difference = abs(recomputed - reported)
        assert difference < 1e-12, f'{corpus} {metric} eşleşmedi'
        test_rows.append({
            'dataset': 'CREMA-D' if corpus == 'cremad' else 'MELD',
            'metric': metric,
            'sample_count': len(predictions),
            'recomputed': recomputed,
            'reported': reported,
            'absolute_difference': difference,
        })
test_comparison = pd.DataFrame(test_rows)
test_comparison

,dataset,metric,sample_count,recomputed,reported,absolute_difference
0,CREMA-D,accuracy,321,0.442368,0.442368,0.0
1,CREMA-D,balanced_accuracy,321,0.441568,0.441568,0.0
2,CREMA-D,macro_f1,321,0.429887,0.429887,0.0
3,CREMA-D,weighted_f1,321,0.428280,0.428280,0.0
4,MELD,accuracy,430,0.190698,0.190698,0.0
5,MELD,balanced_accuracy,430,0.189104,0.189104,0.0
6,MELD,macro_f1,430,0.187391,0.187391,0.0
7,MELD,weighted_f1,430,0.189153,0.189153,0.0


In [4]:
# Held-out test belirsizliği ve validation-temelli kalibrasyon özeti.
evaluation_rows = []
for corpus in CORPORA:
    corpus_dir = ODEV3 / 'outputs' / corpus
    uncertainty = json.loads((corpus_dir / 'test_uncertainty.json').read_text(encoding='utf-8'))
    calibration = json.loads((corpus_dir / 'temperature_scaling.json').read_text(encoding='utf-8'))
    macro_interval = uncertainty['metrics']['macro_f1']
    before = calibration['test']['before']
    after = calibration['test']['after']
    evaluation_rows.append({
        'dataset': 'CREMA-D' if corpus == 'cremad' else 'MELD',
        'test_samples': uncertainty['sample_size'],
        'bootstrap_iterations': uncertainty['iterations'],
        'macro_f1': macro_interval['estimate'],
        'macro_f1_ci_lower': macro_interval['lower'],
        'macro_f1_ci_upper': macro_interval['upper'],
        'temperature': calibration['fit']['temperature'],
        'test_nll_before': before['negative_log_likelihood'],
        'test_nll_after': after['negative_log_likelihood'],
        'test_ece_before': before['expected_calibration_error'],
        'test_ece_after': after['expected_calibration_error'],
        'class_predictions_preserved': calibration['class_predictions_preserved'],
    })
evaluation_summary = pd.DataFrame(evaluation_rows)
evaluation_summary

,dataset,test_samples,bootstrap_iterations,macro_f1,macro_f1_ci_lower,macro_f1_ci_upper,temperature,test_nll_before,test_nll_after,test_ece_before,test_ece_after,class_predictions_preserved
0,CREMA-D,321,2000,0.429887,0.375362,0.479458,2.009574,1.779091,1.435853,0.245503,0.057472,True
1,MELD,430,2000,0.187391,0.149575,0.225636,3.274361,1.876670,1.788027,0.104895,0.013156,True


In [5]:
# Tüm temel MLP hiperparametrelerinin validation-temelli betimsel özeti.
effect_frames = []
for corpus in CORPORA:
    frame = pd.read_csv(ODEV3 / 'hyperparameter_effects' / corpus / 'parameter_effect_overview.csv')
    frame.insert(0, 'dataset', 'CREMA-D' if corpus == 'cremad' else 'MELD')
    effect_frames.append(frame)
hyperparameter_overview = pd.concat(effect_frames, ignore_index=True)
hyperparameter_overview[[
    'dataset', 'parameter_label', 'tested_value_count', 'best_value',
    'best_runs', 'best_mean_macro_f1', 'worst_value', 'worst_runs',
    'worst_mean_macro_f1', 'mean_macro_f1_spread'
]]

,dataset,parameter_label,tested_value_count,best_value,best_runs,best_mean_macro_f1,worst_value,worst_runs,worst_mean_macro_f1,mean_macro_f1_spread
0,CREMA-D,Learning rate,5,0.001,1,0.468498,0.0001,1,0.442402,0.026096
1,CREMA-D,Gizli katman yapısı,6,768-384-192,1,0.475438,256,1,0.423727,0.051711
2,CREMA-D,Batch boyutu,3,128,2,0.469970,64,26,0.457311,0.012658
3,CREMA-D,Patience,5,10,1,0.479429,5,1,0.445976,0.033453
4,CREMA-D,Aktivasyon,3,gelu,2,0.467436,tanh,2,0.429835,0.037601
5,CREMA-D,Batch normalization,2,true,28,0.459037,false,2,0.457108,0.001929
6,CREMA-D,Dropout,5,0.2,1,0.477650,0,1,0.433970,0.043680
7,CREMA-D,Weight decay,2,0,2,0.469046,0.0001,28,0.458184,0.010862
8,MELD,Learning rate,5,0.0006,1,0.241405,0.001,1,0.213737,0.027667
9,MELD,Gizli katman yapısı,6,512-256,25,0.224009,512-256-128,1,0.201773,0.022237


In [6]:
assert len(best_validation) == 2
assert len(test_comparison) == 8
assert test_comparison['absolute_difference'].max() == 0.0
assert evaluation_summary['class_predictions_preserved'].all()
assert len(hyperparameter_overview) == 16
print('DOĞRULAMA TAMAMLANDI')
print('2 seçilmiş validation konfigürasyonu gösterildi.')
print('8 test metriği tahmin CSV dosyalarından birebir yeniden hesaplandı.')
print('Bootstrap, kalibrasyon ve 16 hiperparametre grup özeti yüklendi.')

DOĞRULAMA TAMAMLANDI
2 seçilmiş validation konfigürasyonu gösterildi.
8 test metriği tahmin CSV dosyalarından birebir yeniden hesaplandı.
Bootstrap, kalibrasyon ve 16 hiperparametre grup özeti yüklendi.
